# DCC 2026 Mission 2 — Acoustic Feature Grid GPU 재개

이 노트북은 확인된 Google Drive 구조를 사용합니다. 저장소는 `MyDrive/prelim` 바로가기를 해석해 찾고, 원본 Run은 그 저장소의 `data/processed/mission2_acoustic_feature_grid/20261010T012333Z_8de41728`에서 읽습니다. 새 결과는 프로젝트 안에 넣지 않고 `MyDrive/mission2_grid_runs/<source-run-id>_gpu_resume`에 기록합니다.

기존 Baseline 5개와 F01–F07 Feature cache는 무결성 검증 후 재사용합니다. 새 학습은 TabM·Tabular ResNet·DCNv2·GANDALF의 28개 조합만 GPU에서 순차 실행합니다. FT-Transformer feature 7개는 제외하며 기존 Baseline은 다시 학습하지 않습니다. 매 실험의 checkpoint·prediction·metrics 저장 및 검증이 끝난 뒤 다음 실험으로 넘어갑니다.

Official Validation은 열거나 사용하지 않습니다. Training ZIP 바로가기를 현재 계정에서 열 수 없으면, 검증된 기존 `/content` Training staging이 없는 한 첫 경로 점검에서 멈춥니다.


In [ ]:
from pathlib import Path
import importlib.metadata
import json
import sys
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
MYDRIVE_ROOT = Path('/content/drive/MyDrive')
if not MYDRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'MyDrive mount is unavailable: {MYDRIVE_ROOT}')

# Check the shortcut before resolve(); an inaccessible Drive shortcut must not
# turn into an empty or misleading /content/drive path.
PROJECT_SHORTCUT = MYDRIVE_ROOT / 'prelim'
if not PROJECT_SHORTCUT.is_dir():
    raise FileNotFoundError(f'Expected accessible shared-project shortcut: {PROJECT_SHORTCUT}')
REPO_ROOT = PROJECT_SHORTCUT.resolve(strict=True)
sys.path.insert(0, str(REPO_ROOT / 'src'))
from dcc_data import mission2_acoustic_feature_grid as grid

LAYOUT = grid.resolve_colab_drive_layout(MYDRIVE_ROOT)
assert LAYOUT['repo_root'] == REPO_ROOT
SOURCE_RUN_ROOT = LAYOUT['source_run_root']
TRAINING_ARCHIVE_ROOT = LAYOUT['training_archive_root']
OUTPUT_ROOT = LAYOUT['output_root']
assert SOURCE_RUN_ROOT.name == grid.COLAB_SOURCE_RUN_ID
assert TRAINING_ARCHIVE_ROOT.parts[-2:] == ('raw_original', '대학부 데이터')
assert OUTPUT_ROOT.name == grid.COLAB_OUTPUT_DIR_NAME
assert OUTPUT_ROOT.parent.resolve() == LAYOUT['mydrive_root']
RUN_ROOT = OUTPUT_ROOT / f'{SOURCE_RUN_ROOT.name}_gpu_resume'

# Stop before package installation, extraction, output creation, or model work.
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required; CPU fallback is disabled. Connect a Colab GPU runtime first.')
GPU_NAME = torch.cuda.get_device_name()

STAGE6_ROOT = REPO_ROOT / grid.STAGE6_REFERENCE_RELATIVE
EGEMAPS_88D = REPO_ROOT / grid.EGEMAPS_88D_RELATIVE
TRANSFER_IDENTITY = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
for required_path in (STAGE6_ROOT / 'contract.json', EGEMAPS_88D, TRANSFER_IDENTITY):
    if not required_path.is_file():
        raise FileNotFoundError(f'Required Training-only provenance artifact is missing: {required_path}')

source_ledger = json.loads((SOURCE_RUN_ROOT / 'run_ledger.json').read_text(encoding='utf-8'))
source_cache_manifest = SOURCE_RUN_ROOT / 'feature_cache' / 'manifest.json'
if not source_cache_manifest.is_file():
    raise FileNotFoundError(f'Existing Training feature cache is missing: {source_cache_manifest}')
for model_id in grid.MODEL_IDS:
    baseline_id = f'baseline_{model_id}'
    baseline_dir = SOURCE_RUN_ROOT / baseline_id
    status_path = baseline_dir / 'status.json'
    if source_ledger.get('experiments', {}).get(baseline_id) != 'COMPLETE':
        raise RuntimeError(f'Existing source Baseline is not COMPLETE: {baseline_id}')
    if not status_path.is_file() or json.loads(status_path.read_text(encoding='utf-8')).get('status') != 'COMPLETE':
        raise RuntimeError(f'Existing source Baseline status is missing/invalid: {baseline_id}')
    artifact_manifest = grid._verify_artifact_manifest(baseline_dir)
    if not grid._checkpoint_files_from_manifest(artifact_manifest['files']):
        raise RuntimeError(f'Existing source Baseline checkpoint is missing: {baseline_id}')

RAW_ROOT = Path('/content/mission2_stage13_training_raw')
if not RAW_ROOT.is_dir():
    RAW_ROOT = Path('/content/mission2_grid_training_raw')
if RAW_ROOT.is_dir():
    TRAINING_ARCHIVES = None  # load_training_context verifies the staged source identity
else:
    # Fail before package installation if this account cannot see the shared ZIP folder.
    TRAINING_ARCHIVES = grid.training_archive_paths(TRAINING_ARCHIVE_ROOT)

print('MyDrive:', MYDRIVE_ROOT.resolve())
print('Project shortcut:', PROJECT_SHORTCUT, '->', REPO_ROOT)
print('Existing source Run:', SOURCE_RUN_ROOT)
print('Training archive folder:', TRAINING_ARCHIVE_ROOT,
      '(visible:', TRAINING_ARCHIVE_ROOT.is_dir(), ')')
print('Existing local Training staging:', RAW_ROOT if RAW_ROOT.is_dir() else 'not present')
print('New result parent (not created yet):', OUTPUT_ROOT)
print('GPU:', GPU_NAME, '| CUDA:', torch.version.cuda)


## 고정 패키지 환경

위 경로·권한·GPU gate가 통과한 경우에만 Stage 6/13에서 사용한 고정 모델 패키지를 설치합니다. CUDA를 CPU 버전으로 교체하지 않습니다.


In [ ]:
%pip -q install tabm==0.0.3 rtdl-revisiting-models==0.0.2 rtdl-num-embeddings==0.0.12 pytorch-tabular==1.2.0 opensmile==2.6.0 pyarrow

## Training ZIP staging

`대학부 데이터` 폴더의 12개 ZIP에서 Training manifest가 가리키는 WAV/JSON만 `/content`에 준비합니다. Official Validation 폴더를 탐색하거나 압축 해제하지 않습니다. 이미 있는 로컬 staging은 아래 context identity 검사를 통과해야만 재사용됩니다.


In [ ]:
import numpy as np
import pandas as pd
from dcc_data import mission2_model_family_screen as screen
from dcc_data.mission2_egemaps_colab import prepare_shared_training_raw

if RAW_ROOT.is_dir():
    print('Existing Training staging found; exact source/hash identity is checked next:', RAW_ROOT)
else:
    cache_paths = screen.models.standalone_training_cache_paths(REPO_ROOT)
    training_manifest = pd.read_parquet(
        cache_paths.manifest_parquet,
        columns=['split', 'call_id', 'audio_path', 'json_path'],
        filters=[('split', '==', 'Training')],
    )
    staging_audit = prepare_shared_training_raw(
        training_manifest, list(TRAINING_ARCHIVES), RAW_ROOT,
    )
    print('Training-only staging:', json.dumps(staging_audit, ensure_ascii=False, indent=2))
    del training_manifest


## 기존 고정 Internal Split과 데이터 정합성

기존 Stage 6 split manifest/prediction row identity를 그대로 로드합니다. 새 split은 만들지 않습니다. 이 단계는 59D·88D cache 및 Training WAV staging의 hash·record key·call_id·label을 함께 검증하며, 어느 하나라도 다르면 Run 생성 전에 멈춥니다.


In [ ]:
PRIOR_RUN_ROOTS = grid.stage10_to13_prior_run_roots(REPO_ROOT)
context = grid.load_training_context(
    REPO_ROOT, RAW_ROOT, prior_run_roots=PRIOR_RUN_ROOTS,
    stage6_root=STAGE6_ROOT, egemaps_cache_path=EGEMAPS_88D,
    transfer_identity_path=TRANSFER_IDENTITY,
)
observed_split = {
    'train_rows': len(context.train_idx),
    'validation_rows': len(context.validation_idx),
    'train_calls': len(np.unique(context.call_ids[context.train_idx])),
    'validation_calls': len(np.unique(context.call_ids[context.validation_idx])),
    'call_overlap': len(set(context.call_ids[context.train_idx]) & set(context.call_ids[context.validation_idx])),
    'split_seed': grid.SEED,
    'group_key': 'call_id',
    'official_validation_used': False,
    'verified_prior_models': list(context.prior_predictions.get('predictions', {})),
    'unavailable_prior_artifacts': context.prior_predictions.get('unavailable', []),
}
assert observed_split['train_rows'] == 711_467, observed_split
assert observed_split['validation_rows'] == 177_891, observed_split
assert observed_split['train_calls'] == 23_360, observed_split
assert observed_split['validation_calls'] == 5_840, observed_split
assert observed_split['call_overlap'] == 0, observed_split
print(json.dumps(observed_split, ensure_ascii=False, indent=2))


## 기존 Baseline과 Feature cache 검증 후 새 결과 Run 준비

이 셀은 기존 5개 Baseline의 checkpoint/prediction/metrics/config/scaler를 검증해 새 Drive 결과 경로에 복사합니다. 기존 Feature cache는 원본 hash를 확인하고 메모리에서 재사용하며 재추출하지 않습니다. 결과 경로가 없을 때만 생성합니다.


In [ ]:
RUN_ROOT, resume_audit = grid.prepare_colab_resume_run(
    context, SOURCE_RUN_ROOT, OUTPUT_ROOT,
)
assert RUN_ROOT.parent.resolve() == OUTPUT_ROOT.resolve()
assert RUN_ROOT != SOURCE_RUN_ROOT and SOURCE_RUN_ROOT.is_dir()
assert len(grid.candidate_experiment_ids()) == 28
assert len(context.feature_matrices) == 7
print('Continuation Run:', RUN_ROOT)
print('Source Run (read-only):', SOURCE_RUN_ROOT)
print('Verified baseline/cache reuse:', json.dumps(resume_audit, ensure_ascii=False, indent=2))


## GPU·모델 환경과 28개 입력 계약 preflight

현재 Runtime의 CUDA와 package versions를 실제 기록하고, 모든 Feature schema·행렬 shape/NaN/Inf·Scaler fit scope·모델별 input dimension을 확인합니다. 이 gate가 PASS하기 전에는 fit을 시작하지 않습니다.


In [ ]:
environment = grid.record_environment(RUN_ROOT, device='cuda')
preflight = grid.validate_candidate_preflight(context, RUN_ROOT)
assert environment['status'] == 'READY' and environment['device'] == 'cuda'
assert preflight['status'] == 'PASS' and preflight['candidate_count'] == 28
print('ENVIRONMENT', json.dumps(environment, ensure_ascii=False, indent=2))
print('CANDIDATE PREFLIGHT', json.dumps(preflight, ensure_ascii=False, indent=2))


## GPU 학습 — 28개를 한 번에 하나씩 순차 실행

아래 controller는 병렬 학습을 하지 않습니다. 각 후보 하나만 fit하고 checkpoint·prediction·metrics·hash manifest를 저장한 뒤 검증에 통과하면 다음 후보를 시작합니다. Runtime이 fit 중 끊기면 해당 항목은 RUNNING/부분 artifact로 남으며 자동 재학습·덮어쓰기를 하지 않고 중단합니다. 다시 실행하면 COMPLETE 및 hash 검증을 통과한 이전 결과만 건너뜁니다.


In [ ]:
fit_result = grid.run_remaining_candidate_fits(context, RUN_ROOT, device='cuda')
assert fit_result['status'] == 'ALL_28_COMPLETE_WITH_5_BASELINES_REUSED_7_SKIPPED'
print('ALL CANDIDATE FITS VERIFIED', fit_result['completed_count'], '/', fit_result['candidate_count'])


## 결과 분석과 Stage 14 인계

학습 완료 artifact만 다시 검증한 뒤 재학습한 후보와 기존 Baseline의 증분 효과, Stage 10–13 저장 모델과의 성능/row-level 오류 변화, duration/overlap/confidence 구간을 분석합니다. 기존 Internal Validation의 누적 사용과 28개 후보 비교에 따른 선택 편향을 보고서에 적고 Official Validation은 사용하지 않습니다. Ensemble 실행이나 최종 모델 선정은 하지 않습니다.


In [ ]:
comparison, historical_comparison, slice_comparison, transitions = grid.compare_grid_results(context, RUN_ROOT)
report_path = grid.write_final_report_ko(context, RUN_ROOT)
print('Matched Baseline comparisons:', len(comparison))
print('Historical Stage 10–13 comparisons:', len(historical_comparison))
print('Slice rows:', len(slice_comparison))
print('Changed row transitions:', len(transitions))
print('REPORT_WRITTEN', report_path)
display(comparison)
display(historical_comparison.head(20))
print(report_path.read_text(encoding='utf-8'))
